# Step 2 · LTX-2 baseline batch

Generate one video per prompt from the 50-row dataset. The 49-frame default comes from the duration test.
Settings other than clip length match the duration test. Each completed video has an importable JSON sidecar.

## Prepare Colab

Select an NVIDIA L4 GPU runtime with High RAM. Leave roughly 60 GiB of local scratch for the weights.
In Colab Secrets, enable a Hugging Face read token named `HF_TOKEN` with access to the
[pinned Gemma encoder](https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized).
Upload this notebook by itself. It bundles the runner and exact prompt CSV.
Leave `SOURCE_REPO` set to `None` unless you want to use a separate checkout.
Only job outputs go to `RUNS`. The CUDA environment and checkpoint cache stay on the Colab VM.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
SOURCE_REPO = None  # Optional path to a full checkout.
RUNS = Path("/content/drive/MyDrive/TattooGen/runs")

In [ ]:
# Bundled runner and prompt CSV. Regenerate with notebooks/sync_colab_assets.py.
import base64
import zlib

if SOURCE_REPO is None:
    packed = (
    'eJztXQlv3MaS/iu9WiwySjgT3YexA2w2UfKM5Nle2y+7C0sgesjmDCMOyfCQrGfov29VdTfZvObkKFlkFMQSyT6r'
    '6/y62PxyEGSfT+yQZ/6DGMVPB6/YwS399y7gIeOhy5I8ZNlMMFmG/fLxf4Yn7AfxwH58d8WmIhQJPIhC9ls0SVme'
    'CpdNnqjC91HAJyyMMjGJovt0JNu9DW9Dfx5HScZ4Mo15korihpM+FH/PeDoL/Elx/VsalTWj9Db0kmjOYp5hKabu'
    'v4PLolCaT+IkckSaFrcyMY89Pyh7zPy5wBHdhu9v3r398Prj2/f/y8bs9uAXfzrLEt+5T7+lGePAv3/7w439/ubX'
    '1x9ev31DpcT1lXfhXp9culenzvWJ8E4m4oIL5/LCuTo7O5ocX58eXYjTa6z93zevf/rbxw/VBlzPcU6Oj65Or049'
    'ccw5Pzq/Opu43vm5e3pxxeFP7rrXRw51/7eb739+9/b1m49UFRZueDI8vp4MXfEw9OKrUco9kYkwjZIUy9+8wQG/'
    'p8LTKJoG4tupmM/58HR4fDIZ+tnwdw7/n9lHwzz8Pedh5v9TuEbN6lAvrrxLIc68iXt+fXR1eXZ6wV1+eSncE/fi'
    'xDu9PPVc9+zsiij1/ub71+9uoNqX25DBz+3BTCBBb4G/zo9PLLjx6LuwVnB9eXGF116c4tXJGV6kmZCXZ0d4Oc19'
    'l4eOwDunoyNLN8pz149s8+nl6EjWn9oBfxIJNfLp5PpO300dHlDJY6OdRBS3j0aXWHQeuTzws6ey/KlsORRTkgMb'
    'WGse04SQq4um5G1bhDMc0lyEVORHHqQCq8d+LAI/FLKeFCk7CoWdZnwqbFjFSlvC8VOQLVl64h1fwFP4Sy0WyZ18'
    'BhWHNFR3OJ8bTUSeF0Tc9cOpLJeJzzg4J3JFYrt+em9nCQ9TL0rmcAPo4Ls05KIBZyac+zjyw8wmwYFWSj7EsWBT'
    'UPGhGGhFSop2dJ+JiKPUz6LkCcsqRrMqBcqm6nwIrT1LaXWFx+b8HpYB1NQgBmUB5AX1YaNCsJiX8LlILaZWw3fh'
    '71QI9/CVHI/voV5iR+zfx3Sf/Ts7+frri1P1GH8S7qeC/cqDXNwkSZQMbg8+YMl5nmZsIhiHFkLNDezDf/3iZ4IB'
    'mcRUJKDqDisdyfGwKGGBCAfy6pD9y5guU5HpW4dYhIdPgxBGdI0XIfs3doUlj5lHl36oWjtcPNh/wA1Y4MwPHdU/'
    'c6I8zGAUHqlnXHN2FX5zbDHBnRnjGYyGw+Suy+E/+tmsIOsoimG0tE7AT+PbgzzzhsB2/hS5MhSPyNhjlIZDxlOG'
    '7GIOMXpMQSEEMKQBtDj6wXey94LDkg+w5KHqUa4YlvwCxBxArU+FUPmgnO4OX7HKTeRouD1KQVvHg0OiEhRAOmGX'
    'z8U6IKlV6wXpsQTRHBeJB4EuMHpAUqaDckFiH5lELgL+Dc3rxhavw0e0hB9+lXzjRGHGoWoe+r/ngpEoIPu8/iEl'
    'WwssBRYqe9JtVxiJuJyNURdP4C+lRsrOoQiykjlHvAb5nooBrPL5sZxOKRQtDHlsNNgxo/9UnbNE/J77oDOInc6P'
    'mMszeJQVa4gzggkBJ3wGwTF4sJxWueQ0HhgxWGXhFpOQpaA7KUfdQwcBXr4Obq58lUzAYoQgzqlqsliBQgqyx6gi'
    'NpWlKHkJx1KRZON2wVm1+1LrlCu1eODvgcowXKCgMVSthQoJx7GDvUhBeyNvZpLrykGTdzZmn+7kJbKxD1r0s6W5'
    'WYT5HF05YY7UGNm/svdRBo9xotNslqKXpwaURSyFvjmssZ+k2RD6YsLzhIM84CRRmupKo7I9sEzIK2M5DNByxmqW'
    'pXCcRP9S732SNV/dsW/0nVfy1l2Nd3HOIx6D1nIHXypa5BXO2mI1JfJKc+4neCo9BiGoNP4mJ4X6wzs0qGc10kRk'
    'eRJqZ0cavDTL3SebZJZaJhNlFlBdw7ilaVbSc3vQKesgB3CTnBMRQMUAKEyWstKufPoAvo+yot7twZfSu33+jy91'
    'T/S51gK6HTGNWvpxlYdIVHyEv9UDwyajKrfRdR+gxdD8Q2akNCHQw6TDSEhCKt9/RK25/hTYn4wELcmMn5yDH3Q4'
    'monP6tmhOYDQT2eoQcgREJ9jIFJp+B/Av4mA6WgwOCw7zT3QTzCm0Tw+q9ttKiY+g5CBLWiOk9w6eRcdJ4cn0DQG'
    'KiN0ulIawwgFgxhsoG2c54uAlN2gxiZWjS2Ma+TaBsNazeUuubacyQANmBrgaAqK5148kc7S1PkEN+5I1uAPkjQa'
    '4WFVnOCx2Qj2k4HymabY65fnQ3VXy0i1h/L+XaNVpDQtzMhPycsckOYseYkeSoNWGQDdtwuOWKhIQQpuPpO2nLIo'
    'z+KcrLEX+KiliEGzmQ/mAMNe6Bz6BXs2B6fcpYGkr9gXXM7ngrCKBz4muSjZD/nb1k5+wTI62IwSZ6ZYAIPXWT6d'
    'wnA87sA88yKInXl4ZbvRY4hsBJ5ryON0FmXFLaMNiAJtJ0rEyIwJRhAOyMjFtedz3e4k9wPXjiOY8lOtBT3idJT5'
    'J0DVMiTRlT++PvnVd9+G4gPefafDmK5m8swP0tEkiCCALoJzYt0b6ecvqSm5GgleVP873sIAP11SN3uKRVHrrQyA'
    'sDKukSHbtBojJ3c5sh1/4H7AJ5r3jIfAajZE2D4sksNjPvExLhygxzGAwPW6yXTvwTL484rjkUkDriAUBE4m3LkX'
    'iK5oHwrY7pczNvjw96tr6TeIRwECl8cxeUTsp3f/KK25HA9oEDlOeQl94YCLQsAERQlYz3QwOL6w2PHFoaXqj+Uv'
    'uESSjWXJCdIrg1KjLBrIW3TnyhZn81MvVI0Td9WarzesisoiJT8OYGAWjm6UYRW6b/Mx/daXE30Jomq3Ds9sm9Yp'
    'fQqdWRKF/j+FNAjyOVAZhhmlIxE++PBYqY6//Wh/fPvzzZu6xqcKix2ymxD5hFws3YrCulLhgFYgRYpPITbEhUPM'
    'hanwtlzCOMH45vbgewywUSnJctxBwEq7czFP8FHsQ6TpsjIWlyoJ9a4X5OlsjEpItVvTHgMjxEZ9509HhKRBVR1p'
    'jxthtqTCmP5VzRp9jxt9lN6FVUEHih7qHkdbD4pCGAPU9d2gXI9iOquN3mJz/tl+jJJ7MJDjE8OTgWgvekQNk4kk'
    'TMdgoL4uKIN/kwbSFxVkTd76zQ9/A1m7U03qVSWNNTaU1Qj1lD2PQtC72WxQ0tFihMbZSRRlYzV53YrSaNBQu941'
    'CGJoyjH9a7EgSjjM6K4u52Ul01iMTbtgjA9EU6FHNvYxNjTp6M3bNze1iauRjQoATK9mRe0bA1eDbRPuhoLqHMgP'
    'rz/8XB2IMssF0FaY5gTULQxAP7DQg7WY6aZ2mOnCxDowEdB0GAwi5ggcVdimPMj8vyNm+BM9eMfB2anbKWqDlmsk'
    'HZcHXpjX7/7x8a398fUvr9/8hHyR2bKEM8vD+9SG0Gyy3GQK1+e2H+km5QrIhrQ6RDrIoQ9Iv7b4tO1TMdaNtIGn'
    'kNRCa2todSxDhk8VtPUOxVUq9lqhAnm9s6o9VAHYskIdmIWm03s/thExHh/JgUi3ozoSjQab/RhWorAiGEJndiz4'
    'vT0X8yh5QjcIvX9l9jKeoAZEAz+KReLZFAgCPU3ETLbnh54ApgO/ARe9Fj+kQGaMQ5oCLY0DyswYOLQBdEkgUz2S'
    'nv4dom8VWLqcewOvrhNaAvNlBQ3UQ6ME0ZdPFGKPveVzW7rzahyFb19rnO7biCqUzRDQrxopSUSIv7lkIm5pT0qF'
    '5GDQeciZY8XPRdVyO+DusFa93C/orl7fU4BGQKJA95JGrbanUH5NdIXygzSC0baltR1XJBuN0YRnDoSd4KaMj6vM'
    'qP/0PcUgo5LQGPzUaN2K1FWcT4h5tMVQ4g1exJdG288anybstICbvmB/X8lHX909V0E7U7sMWhZprHqhC/T20iYH'
    'EKl1QbqwVGhG9mycZomEEVq5oKIbxx06c9CYrKVpW1mmw/aV6HYvpRTXYZ+vv1aISQUrqssI7lw5Wc4DuyzTHKcJ'
    'upTbrDZ4hjZE1aDhbXA3o9Cl6i36iA2ltqo2FOdUvDW8CaHfwWEVpEJFiK6Sg76sPfUnterI0UpRFsWg729xP+X0'
    'qNKWtKuIpsG6qqBAAxe2fdgEk5I8VPs78I9mDYtAdD/Mcd/MFsjnWrPKAqP5vesj4yRoqck1RhwIQn87ujc9ZbCk'
    '/5TBgYQDvsX5Qk/KDSzCAlmuDQaChwbao8qZeA9KLTa5HFT/iOCDGogXBeg6IQTBmet7pCMzPV4aIvt+FkUp7kJ5'
    'wDkzVaUUUUQKjU6lVVKb36OPAj0Enjz9AJGng3txA6DYWPZOsFymS9QGTpCAi04diOWgKIXrrWaPPNRWZ/SY+JmQ'
    'dCGaufk8TtXKIgQcZuOTQ/YN7qbC+h8ctjYCEVHAIdKVnWnXExw7jJJKdFuhp0SqTzWMzQDAEaf2Q11KApumVgUt'
    'OIdWED+lykX3X9lfWeyr4VeHz0NMnxjGUlUWUN1Xd6+OTp+HqbyNRhr059CrKVTNYtRVAR8WzIjd4gieK/xIi6tA'
    'tXJ/vdQ/5NR2oM8N0FnN3LxZN3EteLKqVbtvVnyuiEgByapJNlFZ/aNj4l9F4ns+RrwwsEDgVGFeGHkhQVqC3haW'
    'NzhD7wG0DKAG98ryRit6RN8FgdxC0SOqWkNpCbQ7CYTM0SNd4ioaUV4NNqw0I5V8e0Oo48vOFGdzP8gRURqzo5LX'
    'i6lrpu+c6k/K1oS0rasB1wWExxCa+GfZhPEna+iUvnQT/mh4v6me9DxaMf8WraUWgPBtaLA9eJS4dLOiYieJaWP8'
    'MDiEX+TztQyaGKgNOVQxM3tMokwgQCj3hmWzDRVpjphCg5QMa4WLQEzZuMpYzUaA7lHSolz0T6Fk5iKbRa5KTil2'
    'qcpNCaIqPVUpVFZ7g3Ug/1UD+pe7Gnq34VUxz5YGn7smNIqj2NyeaJaD+XDchYOpSxrXmEXq4QUVO4yc7H8VM4c/'
    'smttahby7aKh1BowFeVnR8QZu6FfuBsOQkV+VI03C0XyzZgdt8jswPCbdFmiUYAipXb6ODJEPWekffOv6Bf1AFHS'
    'JGIHM5LdJpaoKKrbA5qR9jalm4i3KeFPO8z4z6AeXZQtF4ptmZ9d0YBtPNm94kpVNHzaFrqQjmgzmOAp/Aj0x5DN'
    'pMLzK/aF2npuBYqJFVpspjIDH/iDcE0LgNLc0gwaeLX4i3dC0J/RJZ+lMfVo1CP2QQhW5SCZeSoo9zRC85pQTulc'
    'qr0yA8wPC4ecUknRg9JppaPvkmmOGXjv6MnAFamT+MTyY9t2IweCDrPqiLuuzVUdoMBwWGzLql0aV0UTziyCeCkd'
    'QyGdUmLjFo9UfqUqPFzSPua7NlsnQBQt2JLaUvy2aECrQouFcD8F6fwGL6g6AdSVdpfRipzIoe92N7ikBQlmmf3D'
    'GnOIisdnJ0uqAp9gTU75EGN0f6ME9DBuzR4sWwMlecMoHCqdsbghnBllEFB79AtbLCBC2kEeG9mJ+HCkPHD6G5Zd'
    '/aUhCVnESFakG5SxWEgZ3YKJmlJWDZCphI6SZU+NULk1PJQy3x2WKZGDQdhk1G1bpqbYNkqgbRdpaFIgb0Mg4YFO'
    'YYPZYlZ5MTtrSr6UrbKVSHosTH6lRDUrvYcWswh8rEkEIQwyg5WD/57YThBlM7AiVhA9mpdUjqAHbCnNngJhzaPM'
    '9ywnCqKE0FcLfS8LOsuiyIbOrBiiZ8vhmPKEebAZ4nj6GsYBTQSIg2L7auMQOcaKHAcUIPaDPcAvL3Ly1MKtKLkx'
    'bJlobXhszXnq5OQyHl8NT64tiJdgsNi0hanrIvMdGFZ4D1OOrXQWJVmqpsuTuZUl3PWxHx5YCY0YZ2Qh1J/PLQr9'
    'LRw7GlcLCAKsmEMzoLGtSUKdYCDoRxYmFmJJait5UpSwvDwIrMLlRzhBetARsBsrhj5US4VBNncRuCYUgCYkXQGc'
    'FaNZIRov3xvgTE9QZh5Y7FHI7UzOcMag2GNZGyw0jyG0CuC5zk6VA6SGJFVG7DuJdVhMTZ/h6IdEEWYQSld1Rew7'
    'NGakHPNTpuApFsn8OJAUsBdEbJCW+YjhTn2aT36DuAOLa8IqW4S8jTd9CAVxikySVk4BrnwMDGlff5qAb+CyHNmc'
    'yXVgmptG7HseQ7CItKYFG+YxmVigvHiUjRFPoVDOgV7Bkx6vnNeI/SxETDdwSZKCVHkIJhU0FSEDOGQYLrZLub6U'
    'yJjNYGDTGcxbNaJFqEp6sJP+FPG2xEfYqewa2k3zBNNVYO0eOcQI+EwKjaWoROmtIsR+oCG4CuR2uBwjesLTkBaK'
    'iQdB2cyuP6dsH1xMFGUiBL0tEp5Ynpj7IUrP6dHw9FrKpJUG/lzKip0GAtqxYh6C3EBHAqUG/g38dG49RoFn4c4r'
    'ArfgvASgE4U1E5iHAEJCIUQpS6AqQUySOYgaKL4IxAx0e+bDJXQ3AS3yQOn8luKQ4mmX+Oihd0gPTUiud0NwcIJN'
    'oaEZD+WMUSr8RKV94uRRPNTMmJwnm4D/fz+EEsNpwp+YokpTPJBMneKhaNoUDkk+GgAo9AzYFCmUVtkFKUfCgiuN'
    'NNWRmxIOSe9W4ZDrwmhdlssGdJ1wEEw1whH7RQBx2AyHBxIoPCwKZQIeIyXnWFa4U1FVNxbm/j7g8OZRmlXlgZGO'
    'D4QULTSMvudjIsj/E1k6taDRJJo+QXupdXY0PLvWvM+ReFPRsEPwK0dYGSw2iA++DQXmLHR5wJUdIjqi7XatRx5g'
    'FgtZFBu7tqJJgEnuFozJAl4iiSIzpItqQVpohIxBdwgSTUUyh+IZ0wTJqfVmgWiukn8q9qegUFO8FMk6JUxTuSFi'
    'ik40AvwbhA48n+BJCZSiaU2ikAPaxKlkCbUuG8jUX8fwnBl+2/nR8PwaZUPZj0JKpNFxZhD4WVgSs4ysNAS/3zQ6'
    '7b5axepUXTVtV8itTSLlICsFtczoLPPZaDIKcgUeMiSFJteUE+C4YSic+1JgcPMxxYeV9ZFuFBGjapRqPlvNKGm6'
    'NcWGCNkiNNR9tcPFnpsSGqKktk9RkkaaKwt7QG5/T67detZLe3Z7o1WTw/PSAbw4+sZyOUhYZ9iE62MRf5EkJqD7'
    'w6XuXxE0Ue2q/7dJ0LTE64NZSE7AqawWK6EE4uBWk0AqWbVZi93CgmAtcROnFVoigtjjQvewGTqt7w0WoRLNbx8q'
    'NSTlouLeGVgDyQxFTEEUTnXApEQGpjSdgUMGfoj06O4jn3k+KOwVHDyCGUzvThunuoO3zGgt9/HqUENdetoDJpxw'
    'S7xEc1fywVTuAJ+Aom1yyiKXr6RaU3I0GTudPiL8S3l8q8EL+xBqmYxdGl6hxCNMNE8FURVcQomZwwOvguVhFgFP'
    '04Vwnrqrw6gWPK+KS/QB6RmgRAek1xVPdYETW4N3ilSdkoS0XewArgBP9ObwrRxc/fUs1FXpy0n0oQsFl3FVCYJr'
    'OC+QsPuK/hxZp3Y8r+nSbYnlGRDErkDwbUE+JN4mGPiu/Lj1IO+9KaoJ03XF3ZMQhdKskjHazJDGx0s4D88rSfBt'
    'CYomEp9C4HQV508iF7vG9gy4oontdSAWyy3RBiBeO6HWRs135OBtCpL/dYzP8ZHhuyGUgHgeRUU1s1Ng3gWiN8dE'
    'HxyL3J9eCu0VFqjdc9spvFfACzVwrz0+2swAbQroVem4Phz+x4N6hfFa28vb26+6PB6X3qAEKyS217ETJUH2EtuL'
    'Z5EI/c/bbO7uANwzUIoGvNcVN22Kr/cA7ykiboqx9+wZboaY/4Us2EnF5TPwhw5UXNozgrUNgG+Wz4GHpxM/cVdx'
    '86j6y2B8dexh5wh5YdE2QfgMOm6GkO/IFVwNH99jfUvF7dRwGCVeYYB9Utd2Auom1Cc4JhNulbfXP85nABbtOF/n'
    'bnAfiPqGSCBRck1E/eWBwH1S3zLJOitdPwlcdGX1mei5BgFBRa7q9en4aykI2FNGnwFWbJnRt6pz14n2AZXWBMx3'
    '481tnmu0N0h1sTmv+H+IYXSl79UB9BLvS7mTCFcDXU9MHX/05wH8CgBjV6l8S52+BSBgB/U2gNX3OOAfJUUXjTcy'
    'EAksoqdFsHmBCD4CA3r0Ekq6BhzY4t3tFAs00IgaGtgdSa29i7sy+mfQbP1c8z8P8rffttpO/C7rb3QQeNGaWFsg'
    '8SXs54IBWGEPeKH/t2FEtfrbHHXcoiOU2sXu7wKwT5JuA9T9hd7z2CdOdAvNVcurG7XcvtakdAP2+00EwdOqiX1N'
    'V293qF8diVg1s2/LrPRNIL+Chhsi5n8yp29vnOpydt1466Mlv68169wE/Fw+naLbj+wmovBpNexPJq83sb9W13Ar'
    '7M+AKdbM8ds6E31D4K9O0Q1T0v88XuQ+ib3jfd+j6vsetQzBtuBMAu4aHcx8YJRV/cOWNz52+sZvgW2skSO4MCBb'
    'HXHfEEYkcq6LuL+Qw7jf6OpJ5o4bWe2VRMLuwynKuCx9pNPUm4K3EAtRoleBE3s+osIIy1bOH9zubIq1AzNFu/7P'
    'p+jyKPvPzv0LGaiTehJ7kUe44DwKIw7z6mj7ooQljMBqSOFODJMRg62UN7jyQRTrxVfeIiy9p3MoVnTiuqKu/aEU'
    'PcvTaUse+8JMwAIVNEOumfiM6BZqNRgGfjGiEnMtOaBCRV0mPri7NPaVEwO3OqBi3QirQcDeT6to9fIWxEr78yrW'
    'kqOzWnr78sRACRHqwMmLWpJpF51ZUXPeXiarfYO8wD7OrlgzagJa/kHnVmzg8u0PquhHAM/r+eyNbMFW4LBMzYAJ'
    '5uki17AKFRq+4W4z2dfMD+zprIq18jKIdLs4p2JLb3GP9y2TmouWnHYjG7AzwbZIxYgcAVrvEbyolkyMRedWVJ2+'
    'F8lqXzEpsM9TK9ZO0yjp2e/RFZu7f/vDK3qStctGQnstf7Az97YE+njozKIWgH3x0RVtQN9Oc9rXSB5cOeNpbWBP'
    '0mq351Zs7vDt8y26JeWqnqBeSRHsTLM1YD7Mv8ZPdCXcz1by6YqzlGp4X+9HVXQfALirgyrWAwFNyvV+XkVf/tw+'
    'C3A7AbtupLIbaYCLUnCrZyiFDkwEIwLHT5zy3IrF3l/lyM0dYn6dJ/1teWzF+ucn1cnU96kVm/t2+3z1pac2H7V/'
    'QaAz86/Ax4sXpgL85PwkQXdk9Q3aDtftpRLW10gF3OoQi3VfqzJo+fIHWGyzobs/sWJrSTyup66vkBwoQfYS5EP+'
    'm4FwbrADvAOUb/PT/vo7tWItoE+RbycnVrzozvBfL6g6PWlJYl98AiDh3AXShx4IJl9X8/sW+3pmUu3Okb7NDgHs'
    '5+yKtXG+kpo7Ob1ic5dwn9bXk7ydNpLZK2l9ndB6CfbRQoGKSUUPiX39432bnA3Y6/kVayOCJUF3fIrFPvVvBwJ1'
    'Vs1Nb0/8M2FzAw9Mw+gRlE0EVsZdwfkzP53TDQj2nZa+Ve7fKp7dItzPpFCfR1f049btz7HoT47OW75U0J7yVwfY'
    'TfQPguZUBAHxiOvzOX6g90+F/21yGOA2J1msiwu2EbD/kyz22OAOJemi/XsE1bS/RXC6xggd4GAEiUHdtL1ev46z'
    't1OEcPMjArfc5u1EBCuUe/GDLV7Oz9ubsbrwXdbfBOlO+Ksh88bXFGGOm7mDuznFYtNjAfveIF70NUUg2c5PsXhh'
    'p/EvGExdtbz3YST+Lf7Gop/4bQcvreHw7Q4FXP8QwJ7OslgbAkQq7uYki5d3+vbmqS5g17UXQpZ86rf27cQUeqeo'
    'fObHq/uD5gv1FfCv/0MsNjknsK/jK9Z/xdeg5h9zdsU2juI+eb3jG8FH9Vc+lp4k2PjcoghEDFprYY6geVrFS70R'
    '3PvZgesfVbEefKgJudvDKvrxC/dbXD0J4HHL2yNt5wl2fYUxjX0gce0QXNOZXHyI5xYfElnrvZFdHCi4bVZhlXRb'
    'JLmvfNj0/sj1LSTlpP1jBp1ZhM2PL0798H4akTYRaadLuOCzO3WAcDO7te6HDf58OYQVQm6Z4b7TN0T2sVhf0nda'
    'f59kxc8GN7/XGDnApe2vCXefb1GHDdfa6Fr9dZLePxi8ET6oKNRvjvuWft/+LIu1xOWs8XbI4jzB5icbQzHJA94F'
    'EbYfaNEOEG60jbXGyyE7/1zwhuCgpOAfkeO+rvO3P8CiH6k7b7xosvAYwPoXGeXHmYiePAzbDuGsuohFRNXqJO7i'
    'LEAD01gjX3DbPPdNMcM6PXf0lcbNYcF9pnuXJF3UXxRpfmvYRN3NjzOCdhQLXxxe8IXGGi647tbW6u+KrJQnuMOT'
    'YBZ+lrFCwR3ktvfmCu5RwJ7E7bLlxZJKPmE3Em+Cgfgn6DwwOskiJLCWktECBa6flbHWuyUrJxP2hMdvhg8a1Nx5'
    'rvuGHuM+PaNbpq7av4TQcaBg23ca4yiOn2AxxXwpWrhSPuH6B2Gs+x2E1ZMJd3FCdCdeaBCy30867lMH/yjpuq6+'
    'SbL0pMC2LzlOcGqpH4BWgfUTq/mK7S8V94kHVo/q3En+4JYAYY1wfae89+Yf7vPfl8nR+VHLmyRd3xKunsheIIWT'
    'HDcwQSctBQsbR2O8KGBogBdbflt4W2SwoNgfkO2+8cbw/viLbYXt4Pn/ALldneM='
    )
    assets = json.loads(zlib.decompress(base64.b64decode(packed)))
    runtime_assets = Path("/content/tattoogen-notebook-assets") / 'prompts'
    runtime_assets.mkdir(parents=True, exist_ok=True)
    for name, content in assets.items():
        (runtime_assets / name).write_text(content, encoding="utf-8")
    RUNNER = runtime_assets / "ltx2_native.py"
    PROMPT_CSV = runtime_assets / 'prompts.csv'
    print("Using the runner and prompt CSV bundled in this notebook.")
else:
    checkout = Path(SOURCE_REPO)
    RUNNER = checkout / "notebooks/ltx2_native.py"
    PROMPT_CSV = checkout / "data/metadata" / 'prompts.csv'
    missing = [str(path) for path in (RUNNER, PROMPT_CSV) if not path.is_file()]
    if missing:
        raise FileNotFoundError("Fix SOURCE_REPO or set it to None. Missing: " + ", ".join(missing))
    print("Using the runner and prompts from", checkout)


## Freeze and preview the 50-prompt baseline

In [ ]:
BASELINE_FRAMES = 49  # Selected from the duration test.
SEED = 42
RUN_GENERATION = False
CONTINUE_ON_ERROR = False

FRAME_COUNTS = [BASELINE_FRAMES]

OUTPUT = RUNS / "ltx2_baseline"
args = [
    str(RUNNER),
    "--phase", "baseline", "--csv", str(PROMPT_CSV),
    "--output", str(OUTPUT), "--seed", str(SEED), "--frames", *map(str, FRAME_COUNTS),
]


preview = subprocess.run([sys.executable, *args], text=True, capture_output=True)
if preview.returncode:
    raise RuntimeError(preview.stderr.strip() or preview.stdout.strip() or "Plan preview failed.")
plan = json.loads(preview.stdout)
print("GPU jobs:", len(plan["jobs"]), "| video frames:", sorted({j["frames"] for j in plan["jobs"]}))
print("Prompt IDs:", list(dict.fromkeys(j["prompt_id"] for j in plan["jobs"])))
print("Output:", OUTPUT)
print(json.dumps(plan["recipe"], indent=2))

## Install the pinned CUDA environment

Check the plan above before running this cell. This uses a separate Python 3.12 environment.

In [ ]:
capability = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True,
).splitlines()[0].strip()
if float(capability) < 8.9:
    raise RuntimeError(f"FP8 requires an L4 or newer supported GPU. Colab assigned SM {capability}.")
print("Free local disk GiB", round(shutil.disk_usage("/content").free / 2**30, 1))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
runtime = Path("/content/tattoogen-ltx2-env")
python = str(runtime / "bin/python")
if not Path(python).exists():
    subprocess.run([uv, "venv", "--python", "3.12", str(runtime)], check=True)
torch_packages = ["torch==2.9.1", "torchvision==0.24.1", "torchaudio==2.9.1"]
pip = [uv, "pip", "install", "--python", python]
subprocess.run([*pip, "--index-url", "https://download.pytorch.org/whl/cu128", *torch_packages], check=True)

revision = plan["recipe"]["code_revision"]
source = Path("/content") / ("ltx2-" + revision[:12])
if not (source / ".git").is_dir():
    subprocess.run(["git", "init", str(source)], check=True)
    subprocess.run(["git", "-C", str(source), "remote", "add", "origin",
                    "https://github.com/Lightricks/LTX-2.git"], check=True)
subprocess.run(["git", "-C", str(source), "fetch", "--depth", "1", "origin", revision], check=True)
subprocess.run(["git", "-C", str(source), "checkout", "--detach", revision], check=True)
constraints = runtime / "constraints.txt"
constraints.write_text(chr(10).join([*torch_packages, "transformers==5.14.1"]) + chr(10))
subprocess.run([
    *pip, "--no-sources", "-c", str(constraints),
    "-e", str(source / "packages/ltx-core"), "-e", str(source / "packages/ltx-pipelines"),
    "huggingface_hub",
], check=True)
subprocess.run([uv, "pip", "check", "--python", python], check=True)

## Generate and save

Set `RUN_GENERATION` to `True` in the plan cell and rerun that cell before starting.
Run one Colab session per output folder. Each completed MP4 and JSON is saved to Drive.
Rerun with the same settings to verify and skip completed jobs. Inspect errors before continuing.

In [ ]:
from google.colab import userdata

if not RUN_GENERATION:
    raise RuntimeError("Review the plan above and set RUN_GENERATION = True to start.")
# Keep credentials in the child process environment. Never print the environment.
worker_env = dict(os.environ, HF_TOKEN=userdata.get("HF_TOKEN"), HF_HOME="/content/ltx2-hf")
# Use True if Xet stalls on this Colab VM.
USE_HTTP_DOWNLOAD = False
if USE_HTTP_DOWNLOAD:
    worker_env["HF_HUB_DISABLE_XET"] = "1"

OUTPUT.mkdir(parents=True, exist_ok=True)
with (OUTPUT / "runtime-packages.txt").open("a", encoding="utf-8") as packages:
    packages.write(chr(10))
    subprocess.run([uv, "pip", "freeze", "--python", python], stdout=packages, check=True)

command = [python, "-u", *args, "--run"]
if CONTINUE_ON_ERROR:
    command.append("--continue-on-error")
with subprocess.Popen(command, env=worker_env, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True, bufsize=1) as worker:
    try:
        for line in worker.stdout:
            print(line, end="", flush=True)
        if worker.wait():
            raise RuntimeError("Generation stopped. Inspect the output above and failures.jsonl, then rerun.")
    except KeyboardInterrupt:
        worker.terminate()
        raise

## Verify coverage before scoring

The table below lists completed videos. Resolve missing prompt IDs or account for the failed runs.
Copy the completed MP4/JSON pairs together. Follow `README_proposed.md` to import, score,
review and export the baseline pairs with the evaluation settings chosen on the development set.
Retain `plan.json`, timing sidecars, and any failures for the study record.

In [ ]:
completed = []
for path in sorted(OUTPUT.glob("*.json")):
    if path.name == "plan.json":
        continue
    row = json.loads(path.read_text())
    completed.append({
        "prompt": row["prompt_id"], "frames": row["settings"]["actual_frames"],
        "seconds": round(row["settings"]["generation_and_export_seconds"], 1),
        "gpu": row["settings"]["gpu"], "sidecar": path.name,
    })
print(f"{len(completed)}/{len(plan['jobs'])} videos saved")
if completed:
    import pandas as pd
    display(pd.DataFrame(completed))
print("Drive output:", OUTPUT)